# Retrato do domicílio típico brasileiro – PNADC 2025

**Pergunta:** Quais são as características mais comuns de um domicílio? (insumo para a documentação da smarthouse)

**Fonte:** PNADC 2025 – 1ª visita (microdados) — grão de **domicílio**.

**Método:** moda ponderada por atributo (`SUM(peso)` = população estimada). Para atributos booleanos: proporção de domicílios com a característica.

**Grão:** a tabela `morador` é pessoa-grão; colapsa-se com `DISTINCT ON (id_domicilio)` para ter 1 linha = 1 domicílio e usar o peso do primeiro morador como proxy do peso domiciliar.

> Complementa o `01_caracteristicas_domicilio.ipynb` (Censo 2010). Os rótulos aqui já estão normalizados pelo ETL da tabela `morador`.

In [ ]:
import sys
from pathlib import Path

import duckdb
import pandas as pd

root = Path.cwd()
while not (root / 'data').exists() and root != root.parent:
    root = root.parent

DB = str(root / 'data' / 'research.duckdb')
con = duckdb.connect(DB, read_only=True)

DOM_VIEW = """
    (SELECT DISTINCT ON (id_domicilio) *
     FROM morador
     WHERE fonte = 'pnadc_2025'
     ORDER BY id_domicilio)
"""


def dist(coluna: str, incluir_nulos: bool = False) -> pd.DataFrame:
    filtro = '' if incluir_nulos else f'WHERE {coluna} IS NOT NULL'
    return con.execute(f"""
        SELECT {coluna} AS valor,
               CAST(SUM(peso) AS BIGINT) AS domicilios,
               ROUND(100 * SUM(peso) / SUM(SUM(peso)) OVER (), 1) AS pct
        FROM {DOM_VIEW} t
        {filtro}
        GROUP BY 1
        ORDER BY domicilios DESC
    """).df()


def pct_sim(coluna: str) -> float:
    return float(con.execute(f"""
        SELECT ROUND(100.0 * SUM(peso) FILTER (WHERE {coluna} IS TRUE)
                     / SUM(peso) FILTER (WHERE {coluna} IS NOT NULL), 1)
        FROM {DOM_VIEW}
    """).fetchone()[0])


In [ ]:
n_dom = int(con.execute(f'SELECT CAST(SUM(peso) AS BIGINT) FROM {DOM_VIEW}').fetchone()[0])
n_linhas = int(con.execute(
    "SELECT COUNT(DISTINCT id_domicilio) FROM morador WHERE fonte = 'pnadc_2025'"
).fetchone()[0])
print(f'Domicilios estimados (Brasil, 2025): {n_dom:,}')
print(f'Domicilios na amostra:               {n_linhas:,}')


## Situação (urbana / rural)

In [ ]:
dist("situacao")

## Atributos numéricos (valor mais frequente)

In [ ]:
numericas = {
    'n_moradores':   'moradores',
    'n_comodos':     'comodos',
    'n_dormitorios': 'dormitorios',
    'n_banheiros':   'banheiros',
}
for col, nome in numericas.items():
    moda = dist(col).iloc[0]
    print(f'{nome:14} -> mais comum: {moda["valor"]}  ({moda["pct"]}% dos domicilios)')


## Abastecimento de água

In [ ]:
dist("agua")

## Esgotamento sanitário

In [ ]:
dist("esgoto")

## Energia elétrica

In [ ]:
dist("energia")

## Equipamentos domésticos e conectividade

Colunas booleanas — proporção de domicílios que possuem.

> **Nota:** `auto_moto` no PNADC cobre automóvel OU motocicleta (pergunta S01031 unificada). No Censo 2010 eram duas variáveis separadas — considerar na comparação 2010 × 2025.

In [ ]:
for col, label in [
    ('geladeira',    'Geladeira'),
    ('maquina_lavar','Maquina de lavar'),
    ('internet',     'Acesso a internet'),
    ('auto_moto',    'Automovel ou moto'),
]:
    p = pct_sim(col)
    print(f'{label:22} -> {p:5.1f}% tem  |  {100-p:5.1f}% nao tem')


## Combustível para cozinhar

Pergunta S01016B — feita apenas a domicílios que preparam alimentos (NULL = não-aplicável, excluído do denominador).

In [ ]:
dist("combustivel_cozinha")

## Próximos passos

- **nb04:** Comparação 2010 × 2025 — evolução das características por atributo.
- Análise de correlação: domicílios com internet também têm geladeira e máquina? (perfil do cliente típico de smarthouse)